In [6]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag
import os

def get_links_and_images(url):
    """
    Собирает ссылки и изображения с заданной веб-страницы.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при получении страницы {url}: {e}")
        return [], []

    soup = BeautifulSoup(response.content, 'html.parser')

    links = []
    for link_tag in soup.find_all('a', href=True):
        href = link_tag['href']
        absolute_url = urljoin(url, href)
        links.append(absolute_url)

    images = []
    for img_tag in soup.find_all('img', src=True):
        src = img_tag['src']
        absolute_img_url = urljoin(url, src)
        images.append(absolute_img_url)

    return links, images

def get_extension_from_url(url):
    """
    Извлекает расширение файла из URL, игнорируя якоря (#).
    """
    try:
        # 1. Убираем якорь (все, что после #) из URL
        url_without_fragment = urldefrag(url)[0]
        parsed_url = urlparse(url_without_fragment)
        path = parsed_url.path
        filename = os.path.basename(path)
        _, ext = os.path.splitext(filename)
        if ext and ext.startswith('.'):
            return ext.lower()
    except Exception as e:
        print(f"Ошибка при извлечении расширения из URL {url}: {e}")
    return None

def download_all_images(image_urls, output_dir="indexed_images"):
    """
    Загружает ВСЕ изображения по URL, определяя расширение из ссылки.
    """
    os.makedirs(output_dir, exist_ok=True)
    downloaded_count = 0
    # Расширяем список всеми возможными форматами изображений
    # Добавляем .svg, .ico, .avif и другие
    image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', 
                        '.svg', '.ico', '.avif', '.tiff', '.tif', '.heic']

    for img_url in image_urls:
        extension = get_extension_from_url(img_url)

        if extension and extension in image_extensions:
            try:
                # Важно: для некоторых сайтов нужен заголовок User-Agent
                headers = {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
                }
                img_response = requests.get(img_url, stream=True, headers=headers)
                img_response.raise_for_status()

                # Проверяем Content-Type на всякий случай (дополнительная защита)
                content_type = img_response.headers.get('content-type', '')
                is_valid_image = any(img_type in content_type for img_type in 
                                   ['image/', 'application/octet-stream'])

                if is_valid_image or extension == '.svg':  # SVG часто отдают как text/xml
                    # Создаем уникальное имя файла, но сохраняем правильное расширение
                    # Используем счетчик для простоты
                    filename = os.path.join(output_dir, f"image_{downloaded_count}{extension}")
                    
                    # Для SVG файлов важно сохранять как текст с правильной кодировкой
                    if extension == '.svg':
                        # Сохраняем SVG как текст с UTF-8
                        with open(filename, 'w', encoding='utf-8') as f:
                            f.write(img_response.text)
                    else:
                        # Для бинарных изображений сохраняем как обычно
                        with open(filename, 'wb') as f:
                            for chunk in img_response.iter_content(chunk_size=8192):
                                f.write(chunk)
                                
                    print(f"✓ Загружено: {img_url} -> {filename}")
                    downloaded_count += 1
                else:
                    print(f"✗ Неверный Content-Type для {img_url}: {content_type}")

            except requests.exceptions.RequestException as e:
                print(f"✗ Ошибка при загрузке {img_url}: {e}")
            except Exception as e:
                print(f"✗ Неизвестная ошибка при обработке {img_url}: {e}")
        else:
            print(f"→ Игнорируем (не изображение): {img_url} (расширение: {extension})")

    print(f"\n✅ Загружено {downloaded_count} изображений в папку '{output_dir}'.")

def save_links(links, output_dir="indexed_data"):
    """
    Сохраняет собранные ссылки в текстовый файл.
    """
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "links.txt"), "w", encoding="utf-8") as f:
        for link in links:
            f.write(link + "\n")
    print(f"✅ Сохранено {len(links)} ссылок в {os.path.join(output_dir, 'links.txt')}")

def main():
    """
    Основная функция веб-индексатора.
    """
    website_url = input("Введите URL веб-сайта для индексации: ")
    if not website_url:
        print("URL не может быть пустым.")
        return

    print(f"🔍 Индексируем: {website_url}...")
    all_links, all_image_urls = get_links_and_images(website_url)

    if all_links:
        save_links(all_links)
    else:
        print("⚠️ Не удалось собрать ссылки.")

    if all_image_urls:
        # Выводим статистику по найденным изображениям
        extensions_found = {}
        for url in all_image_urls:
            ext = get_extension_from_url(url)
            if ext:
                extensions_found[ext] = extensions_found.get(ext, 0) + 1
        
        print(f"\n📊 Найдено изображений: {len(all_image_urls)}")
        print("Статистика по расширениям:")
        for ext, count in extensions_found.items():
            print(f"  {ext}: {count}")
        
        download_all_images(all_image_urls)
    else:
        print("⚠️ Не удалось найти изображения.")

if __name__ == "__main__":
    main()

🔍 Индексируем: https://github.com/masnikovadara47-pixel/python/tree/main...
✅ Сохранено 121 ссылок в indexed_data\links.txt
⚠️ Не удалось найти изображения.
